# 9.2 解码策略 (Decoding Strategies)

解码策略决定了模型如何从概率分布中选择下一个token，直接影响生成质量。

本节涵盖：
- 贪心解码
- 采样解码
- Beam Search
- Speculative Decoding

## 1. 贪心解码与采样解码

**贪心解码**：每步选择概率最高的token，确定性输出，但可能陷入重复。

**采样解码**：按概率分布随机采样，引入多样性。常用变体：
- **Top-K**：只从概率最高的K个token中采样
- **Top-P（Nucleus Sampling）**：从累积概率达到P的最小token集合中采样
- **Temperature**：调整概率分布的平滑度，T>1更随机，T<1更确定

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

logits = torch.randn(1, 50)

greedy_token = logits.argmax(dim=-1).item()

probs = F.softmax(logits, dim=-1)
sample_token = torch.multinomial(probs, 1).item()

top_k = 10
top_k_logits = logits.clone()
top_k_logits[0, top_k_logits.topk(top_k).indices[-1]+1:] = float('-inf')
top_k_probs = F.softmax(top_k_logits, dim=-1)
top_k_token = torch.multinomial(top_k_probs, 1).item()

top_p = 0.9
sorted_probs, sorted_indices = probs.sort(descending=True)
cumulative_probs = sorted_probs.cumsum(dim=-1)
cutoff = cumulative_probs > top_p
cutoff[..., 1:] = cutoff[..., :-1].clone()
cutoff[..., 0] = False
sorted_probs[cutoff] = 0
sorted_probs = sorted_probs / sorted_probs.sum()
top_p_token = sorted_indices[0, torch.multinomial(sorted_probs, 1).item()].item()

for temp in [0.5, 1.0, 2.0]:
    temp_probs = F.softmax(logits / temp, dim=-1)
    entropy = -(temp_probs * temp_probs.log()).sum().item()
    print(f'  T={temp}: entropy={entropy:.2f}')

print('=== Decoding Strategies ===')
print(f'Greedy token: {greedy_token} (highest probability)')
print(f'Random sample token: {sample_token}')
print(f'Top-K (K=10) token: {top_k_token}')
print(f'Top-P (P=0.9) token: {top_p_token}')
print(f'\nTemperature effect (entropy = randomness):')
print(f'\nKey: Greedy is deterministic but repetitive.')
print(f'Top-P is generally preferred over Top-K for better quality-diversity balance.')

## 2. Beam Search与Speculative Decoding

**Beam Search**：维护beam_width个最优候选序列，每步扩展所有候选并保留最优的beam_width个。更全局最优但更慢。

**Speculative Decoding**：用小模型（draft model）快速生成候选token，大模型（target model）并行验证，接受的token直接使用，拒绝的token重新采样。

**Speculative Decoding优势**：
- 数学上保证与直接用大模型采样的分布完全一致
- 小模型猜测准确率高时，可2-3倍加速
- 无损加速，不降低生成质量

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(42)

vocab_size = 50
beam_width = 3
seq_len = 5

logits_seq = [torch.randn(1, vocab_size) for _ in range(seq_len)]

beams = [(0.0, [])]
for step in range(seq_len):
    candidates = []
    for score, tokens in beams:
        log_probs = F.log_softmax(logits_seq[step], dim=-1)
        topk_scores, topk_tokens = log_probs.topk(beam_width)
        for i in range(beam_width):
            new_score = score + topk_scores[0, i].item()
            new_tokens = tokens + [topk_tokens[0, i].item()]
            candidates.append((new_score, new_tokens))
    beams = sorted(candidates, key=lambda x: x[0], reverse=True)[:beam_width]

print('=== Beam Search ===')
print(f'Beam width: {beam_width}')
print(f'Top beams after {seq_len} steps:')
for i, (score, tokens) in enumerate(beams):
    print(f'  Beam {i}: score={score:.2f}, tokens={tokens}')

print(f'\n=== Speculative Decoding ===')
draft_logits = torch.randn(1, 5, vocab_size)
target_logits = torch.randn(1, 5, vocab_size)

draft_tokens = draft_logits.argmax(dim=-1)
n_accepted = 0
for t in range(5):
    draft_prob = F.softmax(draft_logits[0, t], dim=-1)[draft_tokens[0, t]]
    target_prob = F.softmax(target_logits[0, t], dim=-1)[draft_tokens[0, t]]
    accept_prob = min(1.0, (target_prob / (draft_prob + 1e-8)).item())
    if torch.rand(1).item() < accept_prob:
        n_accepted += 1
    else:
        break

print(f'Draft generated 5 tokens, {n_accepted} accepted by target model')
print(f'Effective speedup: draft is ~5x faster, acceptance rate ~{n_accepted/5:.0%}')
print(f'\nKey: Speculative decoding is lossless - output distribution is identical to target model.')